In [29]:
# !pip install -q torch transformers scikit-learn pandas matplotlib tqdm   # (opsional) pasang library jika belum ada

import os                      # operasi path & folder
import gc                      # garbage collector: membebaskan memori antar-fold
import re                      # regular expression untuk membersihkan teks
import html                    # html.unescape: &amp; → &
import random                  # keacakan Python
import contextlib              # nullcontext untuk mode tanpa AMP

import numpy as np             # array numerik
import pandas as pd            # tabel data
import torch                   # inti deep learning
import torch.nn as nn          # nn.CrossEntropyLoss
from torch.utils.data import Dataset, DataLoader         # penyusun batch
from transformers import (
    AutoTokenizer,                                       # tokenizer otomatis sesuai model
    AutoModelForSequenceClassification,                  # model + head klasifikasi
    get_linear_schedule_with_warmup,                     # jadwal LR: warmup lalu turun linear
)
from sklearn.model_selection import StratifiedKFold      # K-Fold yang menjaga proporsi kelas di tiap lipatan
from sklearn.metrics import ConfusionMatrixDisplay, f1_score    # metrik & plot
import matplotlib.pyplot as plt                          # grafik
from tqdm.auto import tqdm                               # bilah kemajuan

import transformers                                      # untuk mencetak versi
print("Torch:", torch.__version__, "| Transformers:", transformers.__version__)   # cetak versi library

Torch: 2.11.0+cu128 | Transformers: 5.14.1


In [30]:
# ===== Konfigurasi utama =====
MODEL_NAMES = ["roberta-base"]              # model yang akan dilatih & di-ensemble (lihat bagian 9 untuk ide model lain)
LOCAL_DIRS = {                              # (opsional) folder lokal format Hugging Face untuk tiap model
    "roberta-base": "../pretrained/roberta_base",   # jika folder ini valid → dipakai; jika tidak → unduh dari Hub
}

SEEDS = [42]                                # daftar seed; tiap seed = satu kali K-Fold lengkap
N_FOLDS = 5                                 # jumlah lipatan cross-validation
EPOCHS = 3                                  # epoch per fold
BATCH_SIZE = 16                             # ukuran batch train
LR = 2e-5                                   # learning rate puncak
WEIGHT_DECAY = 0.01                         # regularisasi L2 (tidak untuk bias & LayerNorm)
WARMUP_RATIO = 0.1                          # 10% langkah awal untuk warmup
MAX_LEN = 96                                # batas token per input (keyword + tweet)
LABEL_SMOOTHING = 0.1                       # ε pada label smoothing
DEBUG = False                               # True → versi mini untuk tes pipeline

if DEBUG:                                   # mode tes cepat
    N_FOLDS, EPOCHS = 2, 1                  #   cukup 2 fold & 1 epoch
    print("⚠️ DEBUG aktif: hasil TIDAK untuk disubmit.")   # peringatan


def set_seed(seed):                         # kunci semua sumber keacakan
    random.seed(seed)                       #   Python
    np.random.seed(seed)                    #   NumPy
    torch.manual_seed(seed)                 #   PyTorch CPU
    torch.cuda.manual_seed_all(seed)        #   PyTorch GPU (aman walau tanpa GPU)

if torch.cuda.is_available():               # GPU NVIDIA tersedia?
    device = torch.device("cuda")           #   pakai GPU
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():   # Apple GPU?
    device = torch.device("mps")            #   pakai MPS
else:                                       # selain itu
    device = torch.device("cpu")            #   CPU (sangat lambat untuk notebook ini)
use_amp = device.type == "cuda"             # fp16 hanya di GPU NVIDIA
print("Perangkat:", device, "| AMP:", use_amp) # tampilkan

Perangkat: cuda | AMP: True


In [31]:
KAGGLE_DIR = "/kaggle/input/nlp-getting-started"     # lokasi data di Kaggle

def find_file(filename):                              # cari file di beberapa folder kandidat
    for folder in [".", KAGGLE_DIR]:                  #   coba folder kerja, lalu folder Kaggle
        path = os.path.join(folder, filename)         #     bentuk path lengkap
        if os.path.exists(path):                      #     ada?
            return path                               #       kembalikan
    raise FileNotFoundError(f"{filename} tidak ditemukan")   # tidak ada di mana pun

df_train = pd.read_csv(find_file("train.csv"))        # data latih
df_test = pd.read_csv(find_file("test.csv"))          # data uji

if DEBUG:                                             # mode tes cepat → kecilkan data
    df_train = df_train.sample(400, random_state=0).reset_index(drop=True)   # 400 baris acak
    df_test = df_test.head(200).reset_index(drop=True)                       # 200 baris pertama

print("Latih:", df_train.shape, "| Uji:", df_test.shape)   # cek ukuran

Latih: (52889, 18) | Uji: (22665, 17)


In [32]:
DROP_COL = ["sumber", "kodekat", "kodesubkat", "kategori", "subkategori", "katakunci", "tanggal", "judul_berita", "nama_tokoh", "jabatan", "organisasi", "lokasi", "alias", "kutipan", "sentimen_kutipan"]
TARGET = "sentimen_berita"
TEXT = "konten_berita"


In [33]:
df_train = df_train.drop(DROP_COL, axis=1)
df_test = df_test.drop(DROP_COL, axis=1)

In [34]:
df_train.head()

,id,konten_berita,sentimen_berita
0,00e3395ef29,Padang (ANTARA) - Oknum Aparatur Sipil Negara ...,Negatif
1,019a47ed0bc,Jakarta -\n\nMahkamah Konstitusi (MK) memasukk...,Negatif
2,01eb3258ed4,Padang (ANTARA) - PT Angkasa Pura II selaku pe...,Negatif
3,02319ba7dbc,"JAKARTA ‚Äì Direktur Utama PT LIB, Akhmad Hadi...",Negatif
4,026dd5917f6,Jakarta -\n\nAda hal baru yang mungkin belum b...,Negatif


In [35]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22665 entries, 0 to 22664
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             22665 non-null  object
 1   konten_berita  22665 non-null  object
dtypes: object(2)
memory usage: 354.3+ KB


In [36]:
df_train[TARGET].unique()

array(['Negatif', 'Netral', 'Positif'], dtype=object)

In [37]:
def normalize_tweet(text):                                    # fungsi pembersih teks minimal
    text = html.unescape(str(text))                           #   &amp; → &  ; str() menjaga agar NaN tak membuat error
    text = re.sub(r"https?://\S+|www\.\S+", "http", text)     #   semua URL (http/https/www) → kata "http"
    text = re.sub(r"@\w+", "@user", text)                     #   semua mention @nama → "@user"
    text = re.sub(r"\s+", " ", text).strip()                  #   rapikan spasi berlebih & spasi di tepi
    return text                                               #   hasil bersih

# --- Lihat dulu contoh normalisasi ---
for t in df_train[TEXT].head(3):                            # ambil 3 tweet pertama
    print("ASLI :", t)                                        #   sebelum
    print("BERSIH:", normalize_tweet(t), "\n")                #   sesudah

ASLI : Padang (ANTARA) - Oknum Aparatur Sipil Negara (ASN) pada Pemprov Sumatera Barat (Sumbar) Yelnazi Rinto yang menjadi terdakwa kasus penyelewengan uang infak Masjid Raya Sumbar serta korupsi sejumlah anggaran lain divonis dengan hukuman tujuh tahun penjara.

"Menyatakan terdakwa terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana korupsi secara berlanjut, menjatuhkan pidana selama tujuh tahun," kata Hakim Ketua Yose Ana Roslinda dalam amar yang dibacakan di Padang, Jumat.

Amar tersebut dibacakan hakim ketua dalam sidang pembacaan putusan di Pengadilan Tipikor Padang. Majelis Hakim juga menjatuhkan pidana denda sebesar Rp350 juta subsider empat bulan kurungan dan membayar uang pengganti sebesar Rp1.754.979.804.

Dengan ketentuan apabila tidak dibayar paling lama satu bulan setelah putusan pengadilan memperoleh kekuatan hukum tetap (inkracht) maka hartanya akan disita dan dilelang.

Baca juga: Oknum ASN jalani sidang perdana kasus penyelewengan infak Masjid Raya

Ba

In [39]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.set_output(transform="pandas")

df_target_encoded = encoder.fit_transform(df_train[[TARGET]])

df_train = pd.concat([df_train.drop(columns=[TARGET]), df_target_encoded], axis=1)

print(df_train)


                id                                      konten_berita  \
0      00e3395ef29  Padang (ANTARA) - Oknum Aparatur Sipil Negara ...   
1      019a47ed0bc  Jakarta -\n\nMahkamah Konstitusi (MK) memasukk...   
2      01eb3258ed4  Padang (ANTARA) - PT Angkasa Pura II selaku pe...   
3      02319ba7dbc  JAKARTA ‚Äì Direktur Utama PT LIB, Akhmad Hadi...   
4      026dd5917f6  Jakarta -\n\nAda hal baru yang mungkin belum b...   
...            ...                                                ...   
52884  b79dee8f99d  Kami akan terus menjadi mitra pemerintah dalam...   
52885  b7a07fc1f97  Bila proposal itu disetujui diharapkan akan le...   
52886  b7a0872bb7e  Jakarta -\n\nWuling Confero lebih dulu mengumu...   
52887  b7a182ae9f4  Jakarta -\n\nKetua Satgas COVID-19, Doni Monar...   
52888  b7a3671d2a1  Kami meyakini keamanan keseluruhan vaksin Sino...   

       sentimen_berita_Negatif  sentimen_berita_Netral  \
0                          1.0                     0.0   
1      

In [42]:
TARGET_ENCODE = ["sentimen_berita_Negatif", "sentimen_berita_Positif", "sentimen_berita_Netral"]

In [46]:
df_train["text_norm"] = df_train[TEXT].map(normalize_tweet)     # teks ternormalisasi untuk train
df_test["text_norm"] = df_test[TEXT].map(normalize_tweet)       # idem untuk test

for target in TARGET_ENCODE:
    grp = df_train.groupby("text_norm")[target]                     # kelompokkan baris berdasarkan teks yang sama
    df_train["dup_n"] = grp.transform("size")                         # berapa kali teks itu muncul
    df_train["dup_mean"] = grp.transform("mean")                      # rata-rata label di kelompok (0=semua 0, 1=semua 1, di antaranya=campur)

    conflict = (df_train["dup_n"] > 1) & (df_train["dup_mean"] > 0) & (df_train["dup_mean"] < 1)   # duplikat DENGAN label berbeda
    tie = conflict & (df_train["dup_mean"] == 0.5)                    # sub-kasus seri 50:50

    print(f"Baris dengan label bertentangan: {conflict.sum()} | di antaranya seri: {tie.sum()}")   # statistik
    df_train.loc[conflict, [TEXT, target]].sort_values(TEXT).head(10)   # tampilkan contoh pertentangan

Baris dengan label bertentangan: 0 | di antaranya seri: 0
Baris dengan label bertentangan: 0 | di antaranya seri: 0
Baris dengan label bertentangan: 0 | di antaranya seri: 0


In [47]:
n_before = len(df_train)                                          # jumlah baris sebelum dibersihkan

df_train.loc[conflict, "target"] = (df_train.loc[conflict, "dup_mean"] > 0.5).astype(int)   # seragamkan ke label mayoritas
df_train = df_train[~tie]                                         # buang kelompok yang seri (ambigu)
df_train = df_train.drop_duplicates(subset="text_norm", keep="first").reset_index(drop=True)   # sisakan 1 baris per teks
df_train["target"] = df_train["target"].astype(int)               # pastikan label bertipe int (bukan float)

print(f"Baris data latih: {n_before} → {len(df_train)}")               # berapa baris yang hilang
print("Proporsi kelas:", df_train["target"].value_counts(normalize=True).round(3).to_dict())   # sebaran kelas setelah bersih

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer